In [1]:
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
import pandas as pd
import pyarrow.dataset as ds


class SimpleStatsReader:
    def __init__(self, input_path, batch_size=1000):
        self.input_path = Path(input_path)
        self.batch_size = batch_size

        if self.input_path.is_file():
            self.parquet_files = [self.input_path]
        elif self.input_path.is_dir():
            self.parquet_files = sorted(self.input_path.glob("*.parquet"))
        else:
            raise FileNotFoundError(f"Path does not exist: {self.input_path}")

        if not self.parquet_files:
            raise FileNotFoundError(f"No parquet files found in: {self.input_path}")

    def get_domain(self, url):
        try:
            domain = urlparse(str(url)).netloc.lower().strip()
            if domain.startswith("www."):
                domain = domain[4:]
            return domain
        except:
            return None

    def describe_values(self, values):
        if len(values) == 0:
            return {"count": 0}

        s = pd.Series(values)
        return {
            "count": int(s.count()),
            "mean": float(s.mean()),
            "median": float(s.median()),
            "min": float(s.min()),
            "p25": float(s.quantile(0.25)),
            "p75": float(s.quantile(0.75)),
            "max": float(s.max()),
        }

    def run(self):
        total_rows = 0
        token_values = []
        language_score_values = []
        text_length_values = []
        domain_counter = Counter()
        null_counter = Counter()

        columns_to_read = ["text", "url", "language_score", "token_count"]

        for parquet_path in self.parquet_files:
            print(f"Processing: {parquet_path.name}", flush=True)

            dataset = ds.dataset(parquet_path, format="parquet")
            scanner = dataset.scanner(columns=columns_to_read, batch_size=self.batch_size)

            for batch in scanner.to_batches():
                df = batch.to_pandas()

                total_rows += len(df)

                null_counter.update(df.isna().sum().to_dict())

                token_vals = pd.to_numeric(df["token_count"], errors="coerce").dropna()
                token_values.extend(token_vals.tolist())

                lang_vals = pd.to_numeric(df["language_score"], errors="coerce").dropna()
                language_score_values.extend(lang_vals.tolist())

                text_lengths = df["text"].dropna().map(lambda x: len(str(x)))
                text_length_values.extend(text_lengths.tolist())

                for url in df["url"].dropna():
                    domain = self.get_domain(url)
                    if domain:
                        domain_counter[domain] += 1

        results = {
            "total_rows": total_rows,
            "null_counts": dict(null_counter),
            "token_stats": self.describe_values(token_values),
            "language_score_stats": self.describe_values(language_score_values),
            "text_length_stats": self.describe_values(text_length_values),
            "top_domains": domain_counter.most_common(20),
        }

        return results

    def print_results(self, results):
        print("\n========== OVERALL ==========")
        print("Total rows:", results["total_rows"])

        print("\n========== NULL COUNTS ==========")
        for k, v in results["null_counts"].items():
            print(f"{k}: {v}")

        print("\n========== TOKEN COUNT STATS ==========")
        for k, v in results["token_stats"].items():
            print(f"{k}: {v}")

        print("\n========== LANGUAGE SCORE STATS ==========")
        for k, v in results["language_score_stats"].items():
            print(f"{k}: {v}")

        print("\n========== TEXT LENGTH STATS ==========")
        for k, v in results["text_length_stats"].items():
            print(f"{k}: {v}")

        print("\n========== TOP DOMAINS ==========")
        for domain, count in results["top_domains"]:
            print(f"{domain}: {count}")

In [4]:
reader = SimpleStatsReader("data/fineweb/")
results = reader.run()
reader.print_results(results)

Processing: 004_00018.parquet
Processing: 004_00019.parquet
Processing: 004_00020.parquet
Processing: 004_00021.parquet
Processing: 004_00022.parquet
Processing: 004_00023.parquet
Processing: 004_00046.parquet
Processing: 004_00047.parquet
Processing: 004_00048.parquet
Processing: 004_00049.parquet

========== OVERALL ==========
Total rows: 1666442

========== NULL COUNTS ==========
text: 0
url: 0
language_score: 0
token_count: 0

========== TOKEN COUNT STATS ==========
count: 1666442
mean: 779.8540597272512
median: 488.0
min: 26.0
p25: 242.0
p75: 915.0
max: 126933.0

========== LANGUAGE SCORE STATS ==========
count: 1666442
mean: 0.9352480293651021
median: 0.9430190920829773
min: 0.6500023603439331
p25: 0.9187495857477188
p75: 0.9620289653539658
max: 0.9995685815811157

========== TEXT LENGTH STATS ==========
count: 1666442
mean: 3640.3669446641406
median: 2267.0
min: 74.0
p25: 1117.0
p75: 4285.0
max: 546489.0

========== TOP DOMAINS ==========
plandisney.disney.go.com: 586
pubmed.ncb

In [5]:
reader = SimpleStatsReader("data/fineweb/004_00018.parquet")
results = reader.run()
reader.print_results(results)

Processing: 004_00018.parquet

========== OVERALL ==========
Total rows: 172447

========== NULL COUNTS ==========
text: 0
url: 0
language_score: 0
token_count: 0

========== TOKEN COUNT STATS ==========
count: 172447
mean: 779.8138326558305
median: 487.0
min: 31.0
p25: 241.0
p75: 916.0
max: 103654.0

========== LANGUAGE SCORE STATS ==========
count: 172447
mean: 0.9352966696017893
median: 0.9430122375488281
min: 0.6500535607337952
p25: 0.9188422560691833
p75: 0.9620256423950195
max: 0.9985013008117676

========== TEXT LENGTH STATS ==========
count: 172447
mean: 3638.0663044297667
median: 2262.0
min: 136.0
p25: 1111.0
p75: 4286.0
max: 481614.0

========== TOP DOMAINS ==========
pubmed.ncbi.nlm.nih.gov: 59
plandisney.disney.go.com: 53
telegraph.co.uk: 46
ibtimes.co.uk: 43
cbsnews.com: 42
latimes.com: 38
sciencedaily.com: 38
prweb.com: 38
timesofindia.indiatimes.com: 37
publications.parliament.uk: 33
techradar.com: 32
foxnews.com: 32
gofundme.com: 32
community.oracle.com: 29
csmonitor.co